In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/name-that-llm/sample_submission.parquet
/kaggle/input/competitions/name-that-llm/train.csv
/kaggle/input/competitions/name-that-llm/test.csv


In [2]:
# Data Loading

train_data = pd.read_csv('/kaggle/input/competitions/name-that-llm/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/name-that-llm/test.csv')

print(f'Train shape: {train_data.shape} | Unique models: {train_data["model"].nunique()} | Prompts: {train_data["prompt_id"].nunique()}')
print(f'Test shape: {test_data.shape}')

train_data.head(5)

Train shape: (2699, 4) | Unique models: 18 | Prompts: 150
Test shape: (3, 2)


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash


In [3]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2699 entries, 0 to 2698
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         2699 non-null   object
 1   prompt_id  2699 non-null   object
 2   text       2699 non-null   object
 3   model      2699 non-null   object
dtypes: object(4)
memory usage: 84.5+ KB


In [4]:
train_data.isnull().sum()

id           0
prompt_id    0
text         0
model        0
dtype: int64

In [5]:
counts = train_data.groupby('prompt_id').size()
print(counts.value_counts())

18    149
17      1
Name: count, dtype: int64


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_union
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score

In [7]:
# test and validation
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(train_data, groups=train_data['prompt_id']))
train = train_data.iloc[train_idx]
val = train_data.iloc[val_idx]

print(train.shape)
print(val.shape)

(2159, 4)
(540, 4)
